# Reset the Entity API database

Clears the **data** tables in SQL Server (the database named in `server_testing/config.json`)
so you can re-run `test_api.ipynb` from a clean state:

- `entities`, `historical_entities`, `entities_location`, `entities_historical_location`
- `entities_validation_results`, `entities_latest_validation_rules`, `entities_validation_summary`
- `entities_responsible_contact`

The **rule-definition** tables (`entities_val_rules`, `entities_val_rule_conditions`,
`entities_val_checks`, `entities_val_logic`) are left untouched, so you do **not** need to
re-seed after a reset.

The notebook uses `entity_api`'s own database module, so it reaches exactly the database the
API uses. `TRUNCATE TABLE` empties a table and restarts its ids at 1; the API server can stay
running.

In [ ]:
import os
import subprocess
import sys

import pandas as pd

# resolve the project root (folder containing entities_run.py); this notebook
# lives one level below it, in entities_notebook_testing/
_here = os.getcwd()
while not os.path.exists(os.path.join(_here, "entities_run.py")) and os.path.dirname(_here) != _here:
    _here = os.path.dirname(_here)
PROJECT_ROOT = _here
sys.path.insert(0, PROJECT_ROOT)  # import entity_api: the API's own database settings

from entity_api.database import connect
from entity_api.sql_server import load_db_config

SEED_SCRIPT = os.path.join(PROJECT_ROOT, "entities_notebook_testing", "seed_validation_rules.py")
# failed-validation logs: same default the app uses (entity_api/config.py)
LOG_DIR = os.path.join(PROJECT_ROOT, os.environ.get("ENTITY_LOG_DIR", "entities_validation_logs"))

DATA_TABLES = ["entities", "historical_entities",
               "entities_location", "entities_historical_location",
               "entities_validation_results", "entities_latest_validation_rules",
               "entities_validation_summary", "entities_responsible_contact"]
RULE_TABLES = ["entities_val_rules", "entities_val_rule_conditions",
               "entities_val_checks", "entities_val_logic"]

_db = load_db_config()
print("project root:", PROJECT_ROOT)
print("database    :", _db.database, "on", _db.server)

In [ ]:
def table_exists(conn, table):
    return conn.execute("SELECT OBJECT_ID(?, 'U')", (table,)).fetchone()[0] is not None


def table_counts(tables):
    rows = []
    with connect() as conn:
        for t in tables:
            # None: the table is not created yet (the API creates it on start)
            n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] if table_exists(conn, t) else None
            rows.append({"table": t, "rows": n})
    return pd.DataFrame(rows)

print("DATA tables (cleared by reset):")
display(table_counts(DATA_TABLES))
print("RULE-definition tables (left untouched):")
display(table_counts(RULE_TABLES))

## Reset — clears data tables only

In [ ]:
def clear_tables(tables):
    # TRUNCATE empties a table and restarts its ids at 1
    cleared = []
    with connect() as conn:
        for t in tables:
            if table_exists(conn, t):
                conn.execute(f"TRUNCATE TABLE {t}")
                cleared.append(t)
    print("cleared:", ", ".join(cleared) if cleared else "(nothing)")

clear_tables(DATA_TABLES)
print()
print("after reset:")
display(table_counts(DATA_TABLES))

## Optional — also wipe the rule-definition tables

Clears `entities_val_rules`, `entities_val_rule_conditions`, `entities_val_checks`, `entities_val_logic`.
After this the engine has **no rules**, so you must run the re-seed cell below
(or `python entities_notebook_testing/seed_validation_rules.py`) before the API can validate anything.

In [ ]:
clear_tables(RULE_TABLES)
print()
print("after reset:")
display(table_counts(RULE_TABLES))

## Optional — re-seed the rule set

Only needed if the `entities_val_*` tables are empty (fresh DB) or you edited `seed_validation_rules.py`.

In [ ]:
out = subprocess.run(
    [sys.executable, SEED_SCRIPT],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)
print(out.stdout or out.stderr)
display(table_counts(RULE_TABLES))

## Optional — delete the failed-validation log files

Removes every `log_*.json` from `entities_validation_logs/`, so the next posts start
with an empty log folder. The database is not touched.

In [ ]:
def clear_logs():
    names = sorted(os.listdir(LOG_DIR)) if os.path.isdir(LOG_DIR) else []
    names = [name for name in names if name.startswith("log_")]
    for name in names:
        os.remove(os.path.join(LOG_DIR, name))
    print(f"deleted {len(names)} log file(s) from {LOG_DIR}")

clear_logs()

## Rebuilding the tables

Nothing here drops a table: the application only creates tables that are missing and never
alters or drops one. To start over with a fresh schema, drop the `entities*` tables in SQL
Server Management Studio; the API creates them again on its next start, and you then need
the re-seed cell above.